# DS 5003, SP 2026 - Healthcare Data Science 

## *For Professor Christian Wernz, PhD, University of Virginia*

---

#### Team Members
* **Robert Ashby** | *University of Virginia, School of Data Science*  gsr3qz@virginia.edu
* **Xavier Colbert** | *University of Virginia, School of Data Science*  kxp3jj@virginia.edu
* **Alysa Pugmire** | *University of Virginia, School of Data Science*  amp3xs@virginia.edu
* **Jasmine Waller** | *University of Virginia, School of Data Science*  vwx5pn@virginia.edu

# CMS HAI Risk Prediction Pipeline
**Objective:** Develop a tool to determine if a facility is at risk of an HAI incident (HAI_1 through HAI_6) and identify which facility measures contribute most to correcting an "at-risk" status.

In [ ]:
import sys
import numpy as np
import importlib
from pathlib import Path

RANDOM_SEED = 42
# Set for numpy, pandas, and sklearn
np.random.seed(RANDOM_SEED)

# Add the 'code' folder to the system path so Python can find your scripts
if './code' not in sys.path:
    sys.path.append('./code')

print("Pipeline initialized. Ready to execute modules.")

# 00: Data Retrieval
Self-healing check: Fetches CMS ZIP if data/raw is empty.

In [ ]:
step_00 = importlib.import_module("00_download_data")
step_00.download_and_extract_cms_data()

# 01: Data Import
Dynamically sweep the `data/raw/` directory and ingest ALL available CMS datasets.

In [ ]:
# Import the module dynamically
step_01 = importlib.import_module("01_data_import")

# Execute the extraction function
raw_datasets = step_01.load_all_raw_data()

# 02: Data Interpretation
Profile all ingested raw datasets to evaluate row counts, memory usage, and the presence of joining keys (Facility ID). This generates the dataset inventory report.

In [ ]:
# Import the module dynamically
step_02 = importlib.import_module("02_data_interpretation")

# Generate the profile and output the CSV to the interim folder
if raw_datasets:
    data_profile = step_02.generate_data_profile(raw_datasets)
    display(data_profile.head(10)) # Preview the top 10 largest files
else:
    print("No data available to profile.")

# 03: Data-Driven Ground Truth Construction

### The Standardized Infection Ratio (SIR)
The predictive target is anchored in the **CMS Standardized Infection Ratio (SIR)**. Unlike raw counts, the SIR is a risk-adjusted summary measure that compares the actual number of infections in a facility to the number of infections predicted based on national baseline data, adjusting for hospital size, type, and patient complexity.


### Risk-Adjusted Threshold as a Binary Label: HAI_at_risk
The ground truth outcome is the specific variable `HAI_at_risk`. A facility is assigned an HAI_at_risk value of 1 if any SIR measure exceeds 1.0. 

* HAI_at_risk = 1 indicates performance worse than the CMS risk-adjusted benchmark on at least one tracked HAI SIR measure.
* By using `HAI_at_risk` as the label, the model trains on standardized performance deviations relative to CMS benchmarks rather than arbitrary thresholds.


### Logic & Data Integrity via Inner Join
1. **Metric Narrowing:** The pipeline isolates SIR-specific measures (`HAI_1_SIR` through `HAI_6_SIR`) to maintain risk-adjusted consistency across different facility scales.
2. **Aggregation (Max Pooling):** Facilities are grouped by `Facility ID`. If a facility exceeds the CMS risk-adjusted benchmark (SIR > 1.0) in any tracked infection category, the `HAI_at_risk` label is set to 1. This identifies systemic vulnerability rather than isolated incidents.
3. **The Inner Join Constraint:** The master modeling set is constructed using an **Inner Join** between the `HAI_at_risk` labels and the general facility features:
    * **Integrity:** Training a supervised model requires known outcomes ($y$). 
    * **Logic:** Facilities with no reported HAI data are excluded because they provide no `HAI_at_risk` label to learn from. This ensures the "Predictive Drivers" are derived entirely from confirmed evidence.
    * **Result:** Every row in the final dataset represents a hospital with a confirmed `HAI_at_risk` status (the label) matched to its corresponding descriptive features (the predictors).

### Population and Distribution
The resulting distribution of the `HAI_at_risk` label represents the population of facilities actively participating in the national reporting program. This provides a data-driven baseline for machine learning algorithms to discover the specific care-process features and metrics that correlate with a facility's "At Risk" status.

In [ ]:
step_03 = importlib.import_module("03_data_processing")

if 'raw_datasets' in locals():
    master_data = step_03.build_target_and_master(raw_datasets)
    display(master_data.head())

# 04: Exploratory Data Analysis
Visualizing the risk distribution across the dataset.

In [ ]:
step_04 = importlib.import_module("04_data_analysis")

if 'master_data' in locals():
    step_04.run_eda()

# 05: Dynamic Feature Discovery and Selection

### 1. The Master Anchor
The process begins with the **Master Modeling Set** (4,789 facilities). By using this as the anchor, the script ensures that every potential predictor is linked to a facility with a known `HAI_at_risk` label.

### 2. Automated File Screening
Instead of manually selecting files, the logic scans all available datasets and applies a **Statistical Gatekeeper** to ensure high-quality inputs:
* **Identity Check:** A file must contain a `Facility ID` to be eligible for the join.
* **Population Overlap (50% Threshold):** Only files that contain data for at least 50% of the target facilities are processed. This automatically filters out noise from specialized datasets or state-level summaries that do not provide enough context for the whole population.

### 3. Structural Reshaping (Long-to-Wide)
CMS data is natively stored in a "Long" format where each measurement occupies a row. Machine learning requires "Wide" data where each row is a unique observation (hospital) and each column is a feature (predictor).
* **Numeric Identification:** The script locates the specific numeric measurement column (e.g., `Score`, `Value`, or `HCAHPS Linear Mean Score`).
* **Pivoting:** The `Measure ID` is transformed into a column header, and the numeric result becomes the cell value. 

### 4. Density-Based Feature Selection
To prevent the model from overfitting on sparse data or "learning from noise," a **Sparsity Filter** is applied to the individual columns:
* If a specific measure (e.g., `MORT_30_PN`) is missing for more than 50% of the hospitals in the modeling set, that feature is discarded.
* This ensures the resulting features are robust, high-density predictors that exist across the majority of the population.

### 5. Final Synthesis and Imputation
The disparate data points—ranging from clinical outcomes to patient satisfaction—are merged via a **Left Join**:
* **Integrity:** The 4,789 facilities are preserved as the primary population.
* **Model Readiness:** Remaining missing values are imputed using the **column median**. This provides a stable baseline for algorithms like XGBoost or Random Forest, ensuring that the "Predictive Drivers" are derived from the central tendency of the evidence.

In [ ]:
step_05 = importlib.import_module("05_feature_identification")

if 'raw_datasets' in locals():
    final_df = step_05.mine_features(
        raw_datasets,
        exclude_files=["Healthcare_Associated_Infections-Hospital"]
    )
    display(final_df.head())

## Data Split and Cross-Validation

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
 
# 1. Setup the Stratified 5-Fold with Shuffling
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
# 2. Separate Features and Target
X = final_df.drop('HAI_at_risk', axis=1)
y = final_df['HAI_at_risk']
 
# 3. Execution Loop
fold_results = []
 
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Verification: Check if the ratio of 'at_risk' remains consistent
    train_ratio = y_train.mean()
    test_ratio = y_test.mean()
    
    print(f"--- Fold {fold + 1} ---")
    print(f"Train At-Risk Ratio: {train_ratio:.2%}, Test At-Risk Ratio: {test_ratio:.2%}")
    print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## Modeling Setup

The next stage builds a consistent modeling workflow across Logistic Regression, Random Forest, and XGBoost. A held-out test split is created for final comparison, while stratified 5-fold cross-validation is retained inside the training data for hyperparameter tuning. Separate preprocessing pipelines are used for linear and tree-based models so that categorical variables, missing values, and feature scaling are handled appropriately.

In [ ]:
# Modeling imports
import importlib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# Train/test split for final held-out evaluation
X = final_df.drop(columns=['HAI_at_risk'])
y = final_df['HAI_at_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED
)

print(f"Training rows: {X_train.shape[0]}")
print(f"Test rows: {X_test.shape[0]}")
print(f"Train at-risk rate: {y_train.mean():.2%}")
print(f"Test at-risk rate: {y_test.mean():.2%}")

# Inner CV used for model tuning on the training set only
cv_inner = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED
)

# Identify column types
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

# Preprocessing for Logistic Regression
numeric_transformer_scaled = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess_logreg = ColumnTransformer([
    ('num', numeric_transformer_scaled, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Preprocessing for Random Forest and XGBoost
numeric_transformer_tree = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

preprocess_tree = ColumnTransformer([
    ('num', numeric_transformer_tree, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Class-balance settings
class_weight_balanced = 'balanced'
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"scale_pos_weight: {scale_pos_weight:.3f}")

# 06: Logistic Regression

Logistic Regression provides the most interpretable baseline in the modeling suite. Because it is a linear model, the preprocessing pipeline includes both imputation and feature scaling. Hyperparameters are tuned with stratified 5-fold cross-validation on the training set, and final performance is reserved for the held-out test set.

In [ ]:
# Step 06: Logistic Regression
step_06 = importlib.import_module("06_model_logreg")
importlib.reload(step_06)

logreg_cv, logreg_pipe, logreg_params = step_06.run_logreg_model(
    preprocess=preprocess_logreg,
    cv=cv_inner,
    Xtrain=X_train,
    ytrain=y_train,
    penalty='elasticnet',
    class_weight=class_weight_balanced,
    random_seed=RANDOM_SEED,
    n_iter=25,
    scoring='roc_auc',
    n_jobs=-1
)

logreg_best = logreg_cv.best_estimator_

print("Best Logistic Regression CV ROC-AUC:", round(logreg_cv.best_score_, 4))
print("Best Logistic Regression Params:")
print(logreg_cv.best_params_)

logreg_results = pd.DataFrame(logreg_cv.cv_results_)
logreg_summary_cols = [
    'rank_test_score',
    'mean_test_score',
    'std_test_score',
    'mean_train_score',
    'param_logreg__C'
]

if 'param_logreg__l1_ratio' in logreg_results.columns:
    logreg_summary_cols.append('param_logreg__l1_ratio')

display(
    logreg_results[logreg_summary_cols]
    .sort_values('rank_test_score')
    .head(10)
)

# 07: Random Forest

Random Forest provides a nonlinear ensemble baseline that can capture interactions and threshold effects without requiring scaled numeric inputs. The same held-out evaluation framework is preserved so that its performance can be compared directly against Logistic Regression and XGBoost.

In [ ]:
# Step 07: Random Forest
step_07 = importlib.import_module("07_model_randforest")
importlib.reload(step_07)

rf_cv, rf_pipe, rf_params = step_07.run_rf_model(
    preprocess=preprocess_tree,
    cv=cv_inner,
    Xtrain=X_train,
    ytrain=y_train,
    class_weight=class_weight_balanced,
    random_seed=RANDOM_SEED,
    n_iter=25,
    scoring='roc_auc',
    n_jobs=-1
)

rf_best = rf_cv.best_estimator_

print("Best Random Forest CV ROC-AUC:", round(rf_cv.best_score_, 4))
print("Best Random Forest Params:")
print(rf_cv.best_params_)

rf_results = pd.DataFrame(rf_cv.cv_results_)
rf_summary_cols = [
    'rank_test_score',
    'mean_test_score',
    'std_test_score',
    'mean_train_score',
    'param_rf__n_estimators',
    'param_rf__max_depth',
    'param_rf__min_samples_split',
    'param_rf__min_samples_leaf',
    'param_rf__max_features'
]

display(
    rf_results[rf_summary_cols]
    .sort_values('rank_test_score')
    .head(10)
)

# 08: XGBoost

XGBoost serves as the strongest gradient-boosted tree baseline in the current workflow. It can accommodate nonlinear relationships and complex feature interactions while still using the same held-out evaluation design. The positive-class weight is scaled from the training data to help address class imbalance during fitting.

In [ ]:
# Step 08: XGBoost
step_08 = importlib.import_module("08_model_xgboost")
importlib.reload(step_08)

xgb_cv, xgb_pipe, xgb_params = step_08.run_xgb_model(
    preprocess=preprocess_tree,
    cv=cv_inner,
    Xtrain=X_train,
    ytrain=y_train,
    scale_pos_weight=scale_pos_weight,
    random_seed=RANDOM_SEED,
    n_iter=25,
    scoring='roc_auc',
    n_jobs=-1
)

xgb_best = xgb_cv.best_estimator_

print("Best XGBoost CV ROC-AUC:", round(xgb_cv.best_score_, 4))
print("Best XGBoost Params:")
print(xgb_cv.best_params_)

xgb_results = pd.DataFrame(xgb_cv.cv_results_)
xgb_summary_cols = [
    'rank_test_score',
    'mean_test_score',
    'std_test_score',
    'mean_train_score',
    'param_xgb__n_estimators',
    'param_xgb__learning_rate',
    'param_xgb__max_depth',
    'param_xgb__min_child_weight',
    'param_xgb__subsample',
    'param_xgb__colsample_bytree'
]

display(
    xgb_results[xgb_summary_cols]
    .sort_values('rank_test_score')
    .head(10)
)

# 09 Held-Out Model Evaluation

The three tuned models are compared on the same held-out test set using ROC-AUC, average precision, balanced accuracy, precision, recall, F1, and Matthews Correlation Coefficient. This keeps model selection and final evaluation separate and provides a clearer comparison than accuracy alone.

In [ ]:
# Helper for held-out evaluation
def evaluate_classifier(model, X_eval, y_eval, model_name, threshold=0.50):
    y_prob = model.predict_proba(X_eval)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    results = {
        'model': model_name,
        'roc_auc': roc_auc_score(y_eval, y_prob),
        'avg_precision': average_precision_score(y_eval, y_prob),
        'balanced_accuracy': balanced_accuracy_score(y_eval, y_pred),
        'precision': precision_score(y_eval, y_pred, zero_division=0),
        'recall': recall_score(y_eval, y_pred, zero_division=0),
        'f1': f1_score(y_eval, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_eval, y_pred)
    }

    print(f"\n===== {model_name} =====")
    print("Confusion Matrix:")
    print(confusion_matrix(y_eval, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_eval, y_pred, zero_division=0))

    return results


model_results = []
model_results.append(evaluate_classifier(logreg_best, X_test, y_test, "Logistic Regression"))
model_results.append(evaluate_classifier(rf_best, X_test, y_test, "Random Forest"))
model_results.append(evaluate_classifier(xgb_best, X_test, y_test, "XGBoost"))

results_df = pd.DataFrame(model_results).sort_values(by='roc_auc', ascending=False)
display(results_df)

# Meeting Discussion 3/27/26

Items to address:

- confusion matrices/code base: Robert
- Tableau visualizations: Alysa
- update data dictionary: Alysa
- start drafting report : Jasmine
    - 3-4 articles framing this as a problem (short lit review)
- start powerpoint : Xavier

target audience is hospital leadership
